In [30]:
import pandas as pd
import re

import numpy as np 
data = pd.read_csv('forecast_data.csv')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [23]:
data.head()

,APFS Number,NAICS,Component,Title,Contract Type,Contract Vehicle,Dollar Range,Small Business Set-Aside,Small Business Program,Contract Status,Contract Number,Contractor,Place of Performance City,Place of Performance State,Primary Contact First Name,Primary Contact Last Name,Primary Contact Phone,Primary Contact Email,Description,Award Quarter,Estimated Solicitation Release,Forecast Published,Forecast Previously Published
0,F2019048487,541330 - Engineering Services,CBP/Air and Marine,MINOTAUR RAPID REACTION AND PROTOTYPING ENGINE...,NaN,Definitive Contract,$10M to $20M,NaN,NaN,"New Requirement, No Contract",NaN,NaN,Washington D.C.,DC,Kenneth,James,(202) 325-4686,Kenneth.t.james@cbp.dhs.gov,This contract will serve to continue the techn...,Q4 2025,04/17/2020,03/09/2020,NaN
1,*F2020051866,"333612 - Speed Changer, Industrial High-Speed ...",USCG/SFLC,Vulkan USA Couplings and Elements,Firm Fixed Price,Indefinite Delivery Contract,$5M to $10M,NaN,NaN,"New Requirement, No Contract",NaN,NaN,Winter Haven,FL,Dan,Kane,(410) 762-6827,Charles.D.Kane@uscg.mil,"The U.S. Coast Guard, Surface Forces Logistics...",Q3 2025,06/09/2025,05/13/2025,02/19/2025
2,F2020052049,541519 - Other Computer Related Services,USCG/CG-914,Procurement of new Biometrics at Sea System,NaN,Definitive Contract,$2M to $5M,NaN,NaN,"New Requirement, No Contract",NaN,NaN,Existing DOD R3 Contact Vehicle,DC,Robert,Mason,(757) 295-2065,Robert.Mason@uscg.mil,The expense is required to support the procure...,Q2 2026,03/01/2021,08/31/2020,NaN
3,F2021052728,713990 - All Other Amusement and Recreation In...,CBP,Firing Range Service,NaN,Blanket Purchase Agreement (BPA),$500K to $1M,NaN,NaN,Follow-on to Existing Contract,HSBP1015A00005,TrailGlades Firing Range,Miami,FL,Nisha,Wills,(305) 849-5199,nisha.Wills@cbp.dhs.gov,"The Department of Homeland Security (DHS), U.S...",Q1 2026,10/28/2025,07/01/2021,10/26/2020
4,*F2021054660,333618 - Other Engine Equipment Manufacturing,USCG/SFLC,ALCO 251 PARTS,Firm Fixed Price,Indefinite Delivery Contract,Over $100M,NaN,NaN,"New Requirement, No Contract",NaN,NaN,Beloit,WI,Mark,Lovingood,(410) 762-6922,mark.a.lovingood2@uscg.mil,The United States Coast Guard (USCG) Surface F...,Q4 2026,03/28/2026,10/04/2024,04/03/2024


In [33]:
data['Contractor'] = data['Contractor'].fillna('New Opportunity')

In [26]:
data['Contractor'].value_counts()

Contractor
New Opportunity                  529
Sikorsky Aircraft Corporation      7
AKIRA TECHNOLOGIES INC             3
Leidos, Inc.                       2
Multiple                           2
                                ... 
BERNAICHE PROPERTY MAINT LLC       1
Anacapa Micro Products             1
Westwind Team LLP                  1
Wit's Solutions Inc                1
Rubbage Control Services LLC       1
Name: count, Length: 357, dtype: int64

In [ ]:
print(data['Contractor'].value_counts().to_string())



In [34]:

def clean_name(name):
    if pd.isna(name):
        return name
    name = name.lower().strip()
    name = re.sub(r'[^a-z0-9 ]', '', name)  # remove punctuation
    name = re.sub(r'\b(inc|llc|corp|corporation|limited|ltd|lp|llp|pllc|gmbh|pc|plc|company|co)\b', '', name)
    name = re.sub(r'\s+', ' ', name)  # collapse multiple spaces
    return name.strip()

data['Contractor_cleaned'] = data['Contractor'].apply(clean_name)


In [43]:
print(data['Contractor_cleaned'].value_counts().sort_index().to_string())



Contractor_cleaned
27 contractors under the current idiq                                                                                                        1
70z0g324foipl0031                                                                                                                            1
aaahcorg                                                                                                                                     1
aaction janitorial service                                                                                                                   1
aalis management consulting                                                                                                                  1
abs group                                                                                                                                    1
absg consulting                                                                                                            

In [58]:
from rapidfuzz import fuzz, process

unique_names = data['Contractor_cleaned'].dropna().unique()
name_map = {}

for name in unique_names:
    if name_map:  # only fuzzy match if there are existing entries
        result = process.extractOne(name, name_map.keys(), scorer=fuzz.token_sort_ratio)
        if result is not None:
            match, score, *_ = result
            if score > 99:
                name_map[name] = name_map[match]
                continue
    name_map[name] = name  # use itself if no good match or first entry

In [64]:
data['Contractor_standardized'] = data['Contractor_cleaned'].map(name_map)
print(data['Contractor_standardized'].value_counts().sort_index().to_string())


Contractor_standardized
27 contractors under the current idiq                                                                                                        1
70z0g324foipl0031                                                                                                                            1
aaahcorg                                                                                                                                     1
aaction janitorial service                                                                                                                   1
aalis management consulting                                                                                                                  1
abs group                                                                                                                                    1
absg consulting                                                                                                       

In [77]:
data['Contractor_original'] = data['Contractor']

def standardize_contractor(name, original):
    if pd.isna(name):
        return name

    name_lc = name.lower()

    if "deloitte" in name_lc:
        return "Deloitte"
    elif "airbus" in name_lc:
        return "Airbus"
    elif "aretec" in name_lc:
        return "AretecSBD"
    elif "ardent" in name_lc:
        return "Ardent Management Consulting"
    elif "booz allen" in name_lc:
        return "Booz Allen Hamilton"
    elif "caci" in name_lc:
        return "CACI"
    elif "corelogic" in name_lc:
        return "CoreLogic"
    elif "ecs" in name_lc:
        return "ECS"
    elif "fcn" in name_lc:
        return "FCN Technology Solutions"
    elif "general dynamics" in name_lc:
        return "General Dynamics"
    elif "human resources research organization" in name_lc:
        return "HumRRO"
    elif "koniag" in name_lc:
        return "Koniag Government Services"
    elif "leidos" in name_lc:
        return "Leidos"
    elif "grumman" in name_lc:
        return "Northrop Grumman"
    elif "padron" in name_lc:
        return "Padron Partners"
    elif "panamerica" in name_lc or "pci" in name_lc:
        return "Panamerica Computers"
    elif "telesolv" in name_lc:
        return "Telesolv Consulting"
    elif "thundercat" in name_lc:
        return "Thundercat Technology"
    elif "v3" in name_lc:
        return "V3Gate"
    elif "widepoint" in name_lc:
        return "WidePoint Corporation"
    elif "abs" in name_lc:
        return "ABS Group"
    elif "aeec" in name_lc:
        return "AEEC Argentys"
    elif "aset" in name_lc:
        return "ASET Corporation"
    elif "akira" in name_lc:
        return "AKIRA Technologies"
    elif "bahfed" in name_lc:
        return "BahFed"
    elif "widepoint" in name_lc:
        return "WidePoint Corporation"
    elif "blue tech" in name_lc:
        return "Blue Tech"
    elif "d&g" in name_lc or "d & g" in name_lc:
        return "D&G Support Services"
    elif "emergent" in name_lc:
        return "Emergent"
    elif "four points" in name_lc:
        return "Four Points Technology"
    elif "govsmart" in name_lc:
        return "GovSmart"
    elif "icf" in name_lc:
        return "ICF"
    elif "lockheed" in name_lc:
        return "Lockheed Martin"
    elif "magnet" in name_lc:
        return "Magnet Forensics"
    elif "paragon" in name_lc:
        return "Paragon Systems"
    elif "snap" in name_lc:
        return "Snap Inc"
    elif "valida" in name_lc:
        return "ValidaTek"
    elif "fy24" in name_lc:
        return "Pending FY24 Award"
    elif "anacapa" in name_lc:
        return "Anacapa Micro Products"
    else:
        return original  



In [78]:
data['Contractor_standardized'] = data.apply(lambda row: standardize_contractor(row['Contractor'], row['Contractor_original']), axis=1)


print(data['Contractor_standardized'].value_counts().sort_index().to_string())


Contractor_standardized
27 contractors under the current IDIQ                                                                                                                                 1
70Z0G324FOIPL0031                                                                                                                                                     1
A-Action Janitorial Service, Inc.                                                                                                                                     1
AAAHC.org                                                                                                                                                             1
ABS Group                                                                                                                                                             2
ACORN SERVICES, INC.                                                                                                                    